In [1]:
import os
import time

import pandas as pd
import numpy as np
from utils import print_c, execution_time, decompress
import json
from tqdm.notebook import tqdm
from scipy.stats import kurtosis, skew
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import itertools
from math import comb

In [2]:
path = r"C:\Users\meghnouh\PycharmProjects\Schizophrenia Detection\features"
param_files = ['DFG_parameters.json', 'preprocessing_parameters.json']

sessions = [os.path.join(path, sess) for sess in os.listdir(path) if sess != "don't read"]
use_x0 = True

In [3]:
def read_param(session, param_files):
    # Reading the JSON files
    param = {}
    for file in os.listdir(session):
        if file in param_files:
            param_path = os.path.join(session, file)
            with open(param_path) as f:
                param.update(json.load(f))

    # Parameters reading
    model_freq = np.array(param['model_freq'])
    n_freq = param['n_freq']
    n_point = param['n_point']
    n_features = param['n_features']
    pars = param['selection'] if param['selection'] is not None else param.get('selection_alpha', None)
    data_case = param.get('data_case', 'evoked')
    alpha = param['alpha']
    version = param.get('version', 0)

    # Channel reading
    channels = param['channel_picks']  # Used channels

    # Printing
    print_c(' Data case: ', highlight=data_case)
    print_c(' Version: ', highlight=str(version))
    print_c(' Alpha: ', highlight=alpha)
    print(' Channels: {:}'.format(param['channel_picks']))
    print(' Model frequencies: {:}'.format(model_freq))
    print(' N_freq = {:}'.format(n_freq))
    print_c(' N_point = ', highlight=n_point)
    print(' Parsimony: {:}'.format(np.array(pars)))
    return param


@execution_time
def read_data(session, use_x0, param):
    for file in os.listdir(session):
        if file != "generated_features.json":  # read only json file containing features not parameters
            continue

        file_path = os.path.join(session, file)
        with open(file_path) as f:
            data: dict = json.load(f)

        dic = []
        for stim in tqdm(data.keys(), position=0):
            for subj, subj_data in tqdm(data[stim].items(), position=1):
                if use_x0:
                    VMS = np.array(subj_data['x0'])  # pre np array shape: List(n_channels)(n_features, n_path)
                else:
                    # subj_feat pre np array shape: List(n_channels)(n_features, n_path)
                    VMS = np.array(decompress(subj_data['features'], n_features=param['n_features']))

                # stim: stim
                # subject ID: subj
                category = subj_data['subject_info']
                channels = param['channel_picks']
                parsimony = np.array(
                    param['selection'] if param['selection'] is not None else param.get('selection_alpha', None))

                for pars_idx, pars in enumerate(parsimony):
                    # VMS[:, :, pars_idx] shape: (n_channel, n_features) per subject
                    features_dict = feature_extraction(VMS[:, :, pars_idx])
                    for feature_name, features in features_dict.items():
                        subj_dict = {'stim': stim,
                                     'subject': subj,
                                     'category': category,
                                     'parsimony': "{:.2f}".format(pars),
                                     'feature': feature_name}
                        # value should have the shape : (n_channels)
                        for ch_idx, channel in enumerate(channels):
                            subj_dict.update({channel: features[ch_idx]})
                        dic.append(subj_dict)

        # columns = ['stim', 'subject', 'category', 'pasimony', *param['channel_picks']]
        data_df = pd.DataFrame(data=dic, index=None)
        return data_df


def feature_extraction(x):
    dic = {'energy': np.sum(x ** 2, axis=1),
           'count_non_zero': np.count_nonzero(x, axis=1),
           'mean': np.mean(x, axis=1),
           'max': np.max(x, axis=1),
           'min': np.min(x, axis=1),
           'pk-pk': np.max(x, axis=1) - np.min(x, axis=1),
           'argmin': np.argmin(x, axis=1),
           'argmax': np.argmax(x, axis=1),
           'argmax-argmin': np.argmax(x, axis=1) - np.argmin(x, axis=1),
           'sum abs': np.sum(np.abs(x), axis=1),
           'var': np.var(x, axis=1),
           'std': np.std(x, axis=1),
           # 'kurtosis': kurtosis(x, axis=1),
           # 'skew': skew(x, axis=1),
           'max abs': np.max(np.abs(x), axis=1),
           'argmax abs': np.argmax(np.abs(x), axis=1),
           'count above val': np.array([np.count_nonzero(row[np.where(row >= 0.05)]) for row in x]),
           'count below val': np.array([np.count_nonzero(row[np.where(row <= -0.05)]) for row in x]),
           'count in range': np.array([np.count_nonzero(row[np.where((row <= 0.5) & (row >= -0.5))]) for row in x]),
           'count out range': np.array([np.count_nonzero(row[np.where((row >= 0.05) | (row <= -0.05))]) for row in x]),
           'count above mean': np.array([np.count_nonzero(row[np.where(row >= np.mean(np.abs(row)))]) for row in x]),
           'count below mean': np.array([np.count_nonzero(row[np.where(row <= np.mean(np.abs(row)))]) for row in x]),
           }

    for key, value in dic.items():
        if value.ndim > 1:
            raise ValueError("Feature {:} not extracted properly, has the dimension {:}".format(key, value.shape))
        if value.shape != (x.shape[0],):
            raise ValueError("Feature not corresponding to the right dimensions")
    return dic


for sess in sessions:
    if sess.split('\\')[-1] == r"don't read":
        continue
    print_c('\nSessions: {:}'.format(sess.split('\\')[-1]), 'blue', bold=True)

    param = read_param(sess, param_files)
    # data_df = read_data(sess, use_x0=use_x0, param=param)

    channels = param['channel_picks']
    parsimony = np.array(param['selection'] if param['selection'] is not None else param.get('selection_alpha', None))
    file_name = os.path.basename(sess) + '-x_0.csv' if use_x0 else os.path.basename(sess) + '.csv'
    # data_df.to_csv(os.path.join(r'C:\Users\meghnouh\PycharmProjects\Schizophrenia Detection\extracted features', file_name))


Sessions: 2022-10-09 23;15
 Data case: evoked
 Version: 1
 Alpha: 0.0008
 Channels: ['Fp1', 'AF7', 'AF3', 'F1', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3', 'FC1', 'C1', 'C3', 'C5', 'T7', 'TP7', 'CP5', 'CP3', 'CP1', 'P1', 'P3', 'P5', 'P7', 'P9', 'PO7', 'PO3', 'O1', 'Iz', 'Oz', 'POz', 'Pz', 'CPz', 'Fpz', 'Fp2', 'AF8', 'AF4', 'AFz', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT8', 'FC6', 'FC4', 'FC2', 'FCz', 'Cz', 'C2', 'C4', 'C6', 'T8', 'TP8', 'CP6', 'CP4', 'CP2', 'P2', 'P4', 'P6', 'P8', 'P10', 'PO8', 'PO4', 'O2']
 Model frequencies: [ 0.1     1.3475  2.595   3.8425  5.09    6.3375  7.585   8.8325 10.08
 11.3275 12.575  13.8225 15.07   16.3175 17.565  18.8125 20.06   21.3075
 22.555  23.8025 25.05   26.2975 27.545  28.7925 30.04   31.2875 32.535
 33.7825 35.03   36.2775 37.525  38.7725 40.02   41.2675 42.515  43.7625
 45.01   46.2575 47.505  48.7525]
 N_freq = 40
 N_point = 615
 Parsimony: [0.05 0.1  0.15 0.2  0.25 0.3  0.35 0.4  0.45 0.5  0.55 0.6  0.65 0.7
 0.75 0.8  0.85 0.9  0.95 1.  ]


In [4]:
for sess in sessions:
    if sess.split('\\')[-1] == r"don't read":
        continue

    if use_x0:
        data_df = pd.read_csv(
            os.path.join(r'C:\Users\meghnouh\PycharmProjects\Schizophrenia Detection\extracted features',
                         os.path.basename(sess) + '-x_0.csv'), index_col=[0])
    else:
        data_df = pd.read_csv(
            os.path.join(r'C:\Users\meghnouh\PycharmProjects\Schizophrenia Detection\extracted features',
                         os.path.basename(sess) + '.csv'), index_col=[0])

category = data_df.groupby(by='subject')['category'].apply('first').to_numpy()

In [5]:
data_df.replace(np.nan, 0, inplace=True)
data_df[data_df.isna().any(axis=1)]

,stim,subject,category,parsimony,feature,Fp1,AF7,AF3,F1,F3,...,CP4,CP2,P2,P4,P6,P8,P10,PO8,PO4,O2


In [6]:
data_df

,stim,subject,category,parsimony,feature,Fp1,AF7,AF3,F1,F3,...,CP4,CP2,P2,P4,P6,P8,P10,PO8,PO4,O2
0,1,1,0,0.05,energy,4.063429,5.688479,3.997397,5.702047,4.402103,...,6.507482,5.802220,15.303707,44.318598,51.474343,120.863775,5.892345,31.861921,28.680174,7.801072
1,1,1,0,0.05,count_non_zero,5.000000,7.000000,6.000000,3.000000,4.000000,...,3.000000,3.000000,3.000000,4.000000,4.000000,2.000000,5.000000,3.000000,4.000000,3.000000
2,1,1,0,0.05,mean,-0.051762,-0.046488,-0.044691,-0.047205,-0.045182,...,-0.044056,-0.047891,0.000463,0.061440,0.084426,0.143657,0.026145,0.052838,0.039477,0.006206
3,1,1,0,0.05,max,0.000000,0.373933,0.000000,0.000000,0.000000,...,0.000000,0.000000,3.103604,6.520573,6.717719,10.981943,2.269512,5.433691,4.404705,2.250224
4,1,1,0,0.05,min,-1.355548,-1.831973,-1.467483,-1.794047,-1.503619,...,-2.312449,-1.998542,-2.229479,-1.120001,-1.591044,0.000000,-0.710998,-1.500248,-2.265125,-1.651391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106915,3,81,1,1.00,count below val,3.000000,4.000000,5.000000,1.000000,1.000000,...,4.000000,2.000000,7.000000,4.000000,4.000000,3.000000,2.000000,2.000000,4.000000,4.000000
106916,3,81,1,1.00,count in range,10.000000,21.000000,19.000000,20.000000,20.000000,...,15.000000,17.000000,17.000000,21.000000,17.000000,22.000000,17.000000,19.000000,15.000000,14.000000
106917,3,81,1,1.00,count out range,10.000000,7.000000,12.000000,8.000000,10.000000,...,11.000000,9.000000,13.000000,7.000000,7.000000,11.000000,7.000000,12.000000,8.000000,6.000000
106918,3,81,1,1.00,count above mean,6.000000,4.000000,9.000000,13.000000,10.000000,...,8.000000,3.000000,1.000000,4.000000,3.000000,12.000000,7.000000,11.000000,5.000000,4.000000


In [7]:
clf = LinearDiscriminantAnalysis(solver='svd', shrinkage=None, priors=[0.5, 0.5], n_components=None,
                                 store_covariance=False, tol=0.0001, covariance_estimator=None)

In [10]:
max_acc = 0
tic = time.time()
grouped = data_df.groupby(by=['stim', 'feature', 'parsimony'])

k_feat = 2
k_chan = 1

for stim in tqdm(data_df['stim'].unique()[[-1]], position=0, desc='Stim'):
    for channel in tqdm(itertools.combinations(channels, k_chan), position=1, desc='Channel', total=comb(len(channels), k_chan)):
        channel = list(channel)
        for pars in parsimony:
            for feature in itertools.combinations(data_df['feature'].unique(), k_feat):
                feature = list(feature)

                if k_chan > 1:
                    if k_feat > 1:
                        pass
                        data = np.hstack([grouped.get_group((stim, f, float("{:.2f}".format(pars))))[channel].to_numpy() for f in feature])
                    else: # k_feat == 1
                        data = grouped.get_group((stim, feature[0], float("{:.2f}".format(pars))))[channel].to_numpy()

                else: # k_chan == 1
                    if k_feat > 1:
                        data = np.array([grouped.get_group((stim, f, float("{:.2f}".format(pars))))[channel[0]].to_numpy() for f in feature]).T
                    else: # k_feat == 1
                        data = grouped.get_group((stim, feature[0], float("{:.2f}".format(pars))))[channel[0]].to_numpy()[:, np.newaxis]

                clf.fit(data, category)
                score = clf.score(data, category)
                if score >= max_acc:
                    max_acc = score
                    print('{:} feature and {:} channel yield {:.2f} %'.format(k_feat, k_chan, max_acc * 100))

print('Best accuracy obtained is {:.2f} % using {:} features and {:} channels'.format(max_acc * 100, k_feat, k_chan))
print('all time', time.time() - tic)

Stim:   0%|          | 0/1 [00:00<?, ?it/s]

Channel:   0%|          | 0/64 [00:00<?, ?it/s]

2 feature and 1 channel yield 56.79 %
2 feature and 1 channel yield 59.26 %
2 feature and 1 channel yield 59.26 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 61.73 %
2 feature and 1 channel yield 62.96 %
2 feature and 1 channel yield 62.96 %
2 feature and 1 channel yield 64.20 %
2 feature and 1 channel yield 64.20 %
2 feature and 1 channel yield 64.20 %
2 feature and 1 channel yield 66.67 %
2 feature and 1 channel yield 66.67 %
2 feature and 1 channel yield 67.90 %
2 feature an